# Leonida bring-up on Colab (UNTESTED notebook)
Does the safe steps only: hardened clone, provenance, scans, Linux staged builds. **Cannot boot GTA VI** (no Windows, no display, limited disk). No game files, keys or tokens belong here.

In [ ]:
!uname -srm; nproc; free -h | sed -n 2p; df -h /content | tail -1
!nvidia-smi -L || echo "no NVIDIA GPU"
!python3 --version

## 1. Get the research repo (upload the ZIP, or clone your published copy)

In [ ]:
from google.colab import files
up = files.upload()   # choose Leonida-PC-Research.zip
!cd /content && python3 -c "import zipfile,os;z=zipfile.ZipFile('Leonida-PC-Research.zip');[(z.extract(i,'.'),os.chmod(i.filename,(i.external_attr>>16)&0o777) if (i.external_attr>>16)&0o777 and not i.is_dir() else None) for i in z.infolist()]"
%cd /content/Leonida-PC-Research
!make check

## 2. Hardened clone of the official upstream (no checkout, hooks off, no submodule init)

In [ ]:
%env GIT_LFS_SKIP_SMUDGE=1
!mkdir -p upstream && git clone --no-checkout --no-recurse-submodules -c core.hooksPath=/dev/null https://github.com/ItsNotRexed/Leonida.git upstream/Leonida

## 3. Record identity BEFORE checkout (paste this output back for the inventory)

In [ ]:
!cd upstream/Leonida && git rev-parse HEAD && git branch --show-current && git log -1 --format='%cI %s' && git tag | tail -5
!cd upstream/Leonida && echo '--- gitlinks' && git ls-tree -r HEAD | awk '$2=="commit"{print $4" -> "$3}' ; echo '--- .gitmodules' ; git show HEAD:.gitmodules 2>/dev/null || echo none
!cd upstream/Leonida && echo '--- LICENSE' && git show HEAD:LICENSE 2>/dev/null | head -20
!cd upstream/Leonida && echo '--- docs/build files' && git ls-tree -r --name-only HEAD | grep -iE 'readme|build|install|\.github/workflows|requirements|\.csproj|CMakeLists' | head -60

**STOP: read the README/build scripts (`git show HEAD:<file>`) before continuing.** Then check out and scan.

In [ ]:
!cd upstream/Leonida && git checkout -q
!bash scripts/static-scan.sh upstream/Leonida && bash scripts/list-vendored.sh upstream/Leonida && bash scripts/submodule-provenance.sh upstream/Leonida
!sed -n '1,60p' reports/scan-Leonida.txt

## 4. Toolchain (Linux). lavapipe = CPU Vulkan, so Vulkan test tools can run without a display

In [ ]:
!apt-get -qq update && apt-get -qq install -y cmake ninja-build vulkan-tools libvulkan-dev mesa-vulkan-drivers glslang-tools spirv-tools dotnet-sdk-8.0
!vulkaninfo --summary 2>&1 | head -15

LLVM 21+ with the AMDGPU target: install from the official LLVM apt repo only after reviewing `llvm.sh`, then check `llc --version | grep amdgpu`.

## 5. Staged build: paste the exact Linux commands from Leonida's README, one stage at a time

In [ ]:
# Template; replace <command> with a documented Linux stage (start with the ~30 GB staged check/shader stages)
# !bash scripts/record-build.sh Leonida -- <command>
!tail -40 BUILD_LOG.md

## 6. Save evidence (Colab disks are temporary)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/leonida-evidence && cp -r reports BUILD_LOG.md /content/drive/MyDrive/leonida-evidence/